In [ ]:
import torch
import copy
import sys
import os
import json
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
from functools import partial
from topic_watermark_processor import TopicWatermarkLogitsProcessor
from semantic_topic_extension import EmbeddingMapper
from model import (
    load_model, 
    load_sentence_model, 
    load_topic_model,
    detect,
)
import numpy as np
from sklearn.metrics import roc_curve, f1_score, auc
from rouge_score import rouge_scorer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = {
    # 'model_name_or_path': 'facebook/opt-2.7b',
#     'model_name_or_path': 'facebook/opt-1.3b',
#     'model_name_or_path': 'facebook/opt-125m',
    'model_name_or_path': 'facebook/opt-6.7b',
#     'model_name_or_path': 'google/gemma-7b',
    'load_fp16' : False,
    'prompt_max_length': None, 
    'max_new_tokens': 50, 
    'generation_seed': 123, 
    'use_sampling': True, 
    'n_beams': 1, 
    'sampling_temp': 0.7, 
    'use_gpu': True, 
    'seeding_scheme': 'simple_1', 
    'delta': 2.0, 
    'ignore_repeated_bigrams': False, 
    'detection_z_threshold': 4.75, 
    'select_green_tokens': True,
    'skip_model_load': False,
    'seed_separately': True,
    'topic_token_mapping': {},
    'granularity': 'kmeans',
}

In [ ]:
topic_list = ["physics", "biology", "mathematics", "chemistry", "computer"]

model, tokenizer = load_model(args)
print("Model loaded")

In [ ]:
dataset_path = './SciTLDRA.jsonl'
samples_list = []
n = 50
with open(dataset_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= n:
            break
        entry = json.loads(line)
        abstract_sentences = entry.get('source', [])
        reference_tldr = entry.get('target', [''])[0].strip()
        title = entry.get('title', '').strip()

        # Skip if missing data
        if not abstract_sentences or not reference_tldr:
            continue

        abstract_text = " ".join(abstract_sentences)
        prompt = f"Summarize the following abstract in one sentence:\n\n{abstract_text}"

        samples_list.append({
            'prompt': prompt,
            'reference_tldr': reference_tldr,
            'title': title
        })


In [ ]:
sentence_model = load_sentence_model()
print("Sentence model loaded")
embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)
total_tokens, vocab_embeddings = embedding_mapper.get_model_vocab_embeddings()
topic_embeddings = embedding_mapper.get_defined_topic_list_embeddings(topic_list)
topic_token_mapping = embedding_mapper.map_tokens_to_topics(total_tokens, vocab_embeddings, topic_list, topic_embeddings, threshold=0.5)
args['topic_token_mapping'] = topic_token_mapping

In [ ]:
def generate_no_watermark(prompt, args, model=None, tokenizer=None):
    gen_kwargs = {
        'max_new_tokens': args['max_new_tokens'],
        'min_new_tokens': args['min_new_tokens']
    }

    if args['use_sampling']:
        gen_kwargs.update(dict(
            do_sample=True, 
            top_k=0,
            temperature=args['sampling_temp'],
        ))
    else:
        gen_kwargs.update(dict(
            num_beams=args['n_beams']
        ))

    # generate without the watermark 
    generate_without_watermark = partial(
        model.generate,
        **gen_kwargs
    ) 

    if args['prompt_max_length']:
        pass
    elif hasattr(model.config,"max_position_embeddings"):
        args['prompt_max_length'] = model.config.max_position_embeddings - args['max_new_tokens']
    else:
        args['prompt_max_length'] = 2048 - args['max_new_tokens']

    tokenized_input = tokenizer(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=True, 
        truncation=True, 
        max_length=args['prompt_max_length']
    ).to(device)

    torch.manual_seed(args['generation_seed'])

    output_without_watermark = generate_without_watermark(**tokenized_input)

    if args['decoder']:
        # need to isolate the newly generated tokens
        output_without_watermark = output_without_watermark[:,tokenized_input["input_ids"].shape[-1]:]

    decoded_output_without_watermark = tokenizer.batch_decode(output_without_watermark, skip_special_tokens=True)[0]

    return decoded_output_without_watermark

def generate_TBW(prompt, detected_topics, args, model=None, tokenizer=None, sentence_model=None):

    detected_topic = ''
    for topic in detected_topics:
        if topic.lower() in args['topic_token_mapping']:
            detected_topic = topic.lower()
            print(f"Topic detected in one to one mapping: {detected_topic}")
            break
    if detected_topic == '':
        embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)

        if args['granularity'] == 'kmeans':
            print(f"Mapping topic with granularity Kmeans.")
            detected_topic, _ = embedding_mapper.kmeans_detected_topics_to_embeddings(
                list(args['topic_token_mapping'].keys()), 
                detected_topics,
            )
        else:
            print(f"Mapping topic with granularity averaging")
            detected_topic, _ = embedding_mapper.detected_topics_to_embeddings(
                list(args['topic_token_mapping'].keys()), 
                detected_topics
            )

    print(f"Detected topic: {detected_topic}")
    watermark_processor = TopicWatermarkLogitsProcessor(
        vocab=list(tokenizer.get_vocab().values()),
        delta=args['delta'],
        seeding_scheme=args['seeding_scheme'],
        select_green_tokens=args['select_green_tokens'],
        topic_token_mapping=args['topic_token_mapping'],
        detected_topic=detected_topic,
    )
  
    gen_kwargs = {
        'max_new_tokens': args['max_new_tokens'],
        'min_new_tokens': args['min_new_tokens']
    }

    if args['use_sampling']:
        gen_kwargs.update(dict(
            do_sample=True, 
            top_k=0,
            temperature=args['sampling_temp'],
#             repetition_penalty=1.1,
        ))
    else:
        gen_kwargs.update(dict(
            num_beams=args['n_beams']
        ))

    generate_with_watermark = partial(
        model.generate,
        logits_processor=LogitsProcessorList([watermark_processor]), 
        **gen_kwargs
    )

    if args['prompt_max_length']:
        pass
    elif hasattr(model.config,"max_position_embeddings"):
        args['prompt_max_length'] = model.config.max_position_embeddings - args['max_new_tokens']
    else:
        args['prompt_max_length'] = 2048 - args['max_new_tokens']

    tokenized_input = tokenizer(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=True, 
        truncation=True, 
        max_length=args['prompt_max_length']
    ).to(device)

    output_with_watermark = generate_with_watermark(**tokenized_input)

    if args['decoder']:
        # need to isolate the newly generated tokens
        output_with_watermark = output_with_watermark[:,tokenized_input["input_ids"].shape[-1]:]

    decoded_output_with_watermark = tokenizer.batch_decode(output_with_watermark, skip_special_tokens=True)[0]

    return decoded_output_with_watermark

In [ ]:
schemes = {
    "NW": generate_no_watermark,
    "TBW": generate_TBW,
}

warmup_length = 10
first_prompt = samples_list[0]['prompt']
args_warmup = copy.deepcopy(args)
args_warmup['max_new_tokens'] = warmup_length
args_warmup['min_new_tokens'] = warmup_length

for scheme, func in schemes.items():
    if scheme == 'TBW':
        detected_topics = load_topic_model(first_prompt)
        _ = func(first_prompt, detected_topics, args_warmup, model=model, tokenizer=tokenizer, sentence_model=sentence_model)
    else:
        _ = func(first_prompt, args_warmup, model=model, tokenizer=tokenizer)

print("Warmup done")

gen_new_tokens = args['max_new_tokens']
args_gen = copy.deepcopy(args)
args_gen['max_new_tokens'] = gen_new_tokens + 5
args_gen['min_new_tokens'] = gen_new_tokens - 5
for sample in samples_list:
    prompt = sample['prompt']
    sample['generated'] = {}

    for scheme, func in schemes.items():
        if scheme == 'TBW':
            detected_topics = load_topic_model(prompt)
            generated_text = func(prompt, detected_topics, args_gen, model=model, tokenizer=tokenizer, sentence_model=sentence_model)
        else:
            generated_text = func(prompt, args_gen, model=model, tokenizer=tokenizer)
        print(f"{scheme}: {generated_text}")
        sample['generated'][scheme] = generated_text

In [ ]:
detected_results = {}

for scheme in ["NW", "TBW"]:
    print(f"Running detection for scheme: {scheme}")
    detection_results = []

    for sample in samples_list:
        prompt = sample['prompt']
        generated_text = sample['generated'][scheme]
        result_formatted = detect(prompt, generated_text, args_gen, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
        is_watermarked = result_formatted[6][1] == 'Watermarked'
        score = float(result_formatted[3][1])
        detection_results.append({
            'is_watermarked': is_watermarked,
            'score': score
        })
        
        print(f"{scheme}: {detection_results}")

    detected_results[scheme] = detection_results

In [ ]:
def get_tpr_f1_at_fpr_levels(fpr, tpr, thresholds, scores, labels, fpr_levels=[0.01, 0.1]):
    results = {}
    fpr_tpr_thresholds = list(zip(fpr, tpr, thresholds))
    fpr_tpr_thresholds.sort(key=lambda x: x[0])  # sort by fpr ascending
    fpr_sorted, tpr_sorted, thresh_sorted = zip(*fpr_tpr_thresholds)
    
    fpr_sorted = np.array(fpr_sorted)
    tpr_sorted = np.array(tpr_sorted)
    thresh_sorted = np.array(thresh_sorted)
    
    for level in fpr_levels:
        idx_candidates = np.where(fpr_sorted >= level)[0]
        if len(idx_candidates) == 0:
            idx = len(fpr_sorted) - 1
        else:
            idx = idx_candidates[0]
        
        chosen_threshold = thresh_sorted[idx]
        chosen_tpr = tpr_sorted[idx]
        
        preds = (scores >= chosen_threshold).astype(int)
        chosen_f1 = f1_score(labels, preds)
        
        results[level] = {
            "threshold": chosen_threshold,
            "tpr": chosen_tpr,
            "f1": chosen_f1
        }
    
    return results


In [ ]:
nw_results = detected_results["NW"]
tbw_results = detected_results["TBW"]
nw_scores = np.array([r["score"] for r in nw_results])
tbw_scores = np.array([r["score"] for r in tbw_results])
combined_scores = np.concatenate([nw_scores, tbw_scores])
combined_labels = np.concatenate([np.zeros_like(nw_scores), np.ones_like(tbw_scores)])

fpr, tpr, thresholds = roc_curve(combined_labels, combined_scores)
roc_auc = auc(fpr, tpr)

print(f"Combined ROC AUC: {roc_auc:.3f}")

tpr_f1_results = get_tpr_f1_at_fpr_levels(fpr, tpr, thresholds, combined_scores, combined_labels, fpr_levels=[0.01, 0.1])
for level, metrics in tpr_f1_results.items():
    print(f"At FPR {level*100:.1f}%: Threshold = {metrics['threshold']:.3f}, TPR = {metrics['tpr']:.3f}, F1 = {metrics['f1']:.3f}")

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_results = {
    "NW": {"rouge1": [], "rouge2": [], "rougeL": []},
    "TBW": {"rouge1": [], "rouge2": [], "rougeL": []}
}

for sample in samples_list:
    reference = sample['reference_tldr']
    for scheme in ["NW", "TBW"]:
        generated = sample['generated'][scheme]
        scores = scorer.score(reference, generated)
        rouge_results[scheme]['rouge1'].append(scores['rouge1'].fmeasure)
        rouge_results[scheme]['rouge2'].append(scores['rouge2'].fmeasure)
        rouge_results[scheme]['rougeL'].append(scores['rougeL'].fmeasure)

rouge_stats = {}
for scheme in rouge_results:
    rouge_stats[scheme] = {}
    for metric in rouge_results[scheme]:
        scores = np.array(rouge_results[scheme][metric])
        mean_score = scores.mean()
        std_score = scores.std()
        rouge_stats[scheme][metric] = {"mean": mean_score, "std": std_score}

print("Average ROUGE Scores and Standard Deviations:")
for scheme, metrics in rouge_stats.items():
    print(f"\nScheme: {scheme}")
    for metric, stats in metrics.items():
        print(f"  {metric.upper()}: Mean = {stats['mean']:.3f}, Std = {stats['std']:.3f}")